# Orthomosaic — Part 2: Colour Correction

**Apply per-channel percentile stretch to the raw orthomosaic to remove sensor-induced colour cast.**

The GeoBrix RasterX function `rx.rst_percentile_stretch` clips each RGB band to its
[2nd, 98th] percentile range of valid (non-black) pixels and rescales to uint8 — a
persisted data-engineering step, so the served COG/PMTiles carry the correction. This
removes the characteristic "hot-red" cast common in GoPro aerial imagery.

> **Prerequisite.** `01a_sfm_orthomosaic` must have written `orthomosaic.tif` to `output_dir`.

> **Runtime.** Runs on **Serverless environment 5**.

---

**Last Update:** September 24, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: Per-channel percentile stretch

In [ ]:
import os
import time as _t_cc
from pathlib import Path as _P
import shutil as _sh
from databricks.labs.gbx import pyrx as _pyrx

_t0 = _t_cc.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* orthomosaics under {output_dir} — run 01 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = ortho_input(grp)
        _sig = _pyrx.input_signature({"grp": grp,
                "up": (_mani.get_sig("dense", f"{grp}::0") or _mani.get_sig("mosaic", grp) or ""),
                "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0)})
        if _pyrx.checkpoint_skip(_mani, "corrected", grp, _sig, force=bool(FORCE_CORRECTED)):
            print(f"[corrected][skip] group {grp!r} — checkpointed ({_gp['corrected']})")
            continue
        # GeoBrix rx.rst_percentile_stretch — per-band 2–98% contrast stretch to uint8,
        # a persisted data-engineering step (the served COG/PMTiles carry the correction).
        _cc_dir = f"{_P(_gp['corrected']).parent}/_corrected_{grp}"
        _P(_cc_dir).mkdir(parents=True, exist_ok=True)
        _content = _P(ortho_input(grp)).read_bytes()
        (
            spark.createDataFrame([("orthomosaic_corrected", _content)], ["source", "content"])
                 .select("source", rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
                 .select("source", rx.rst_percentile_stretch("tile", F.lit(2.0), F.lit(98.0)).alias("tile"))
                 .write.format("gtiff_gbx").mode("overwrite").save(_cc_dir)
        )
        _written = sorted(_P(_cc_dir).glob("*.tif"))
        if not _written:
            raise RuntimeError(f"group {grp!r}: rst_percentile_stretch produced no .tif under {_cc_dir}")
        if _written[0].resolve() != _P(_gp["corrected"]).resolve():
            _sh.move(str(_written[0]), _gp["corrected"])
        _mani.mark_done("corrected", grp, _sig, _gp["corrected"])
        print(f"  group {grp!r}: colour-corrected → {_gp['corrected']}")
    print(f"Colour correction done in {_t_cc.perf_counter()-_t0:.1f}s ({len(_groups)} group(s))")
except Exception as e:
    print(f"[ERROR] Colour correction failed after {_t_cc.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: Before / After Comparison

In [ ]:
import matplotlib.pyplot as plt

# VizX renders the decimated, percentile-stretched raster straight into caller-provided
# axes (ax=), so before/after is one figure — no manual thumbnailing/`imshow`.
_gp0 = group_paths(discover_groups()[0])
_g = _gp0["group"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
vz.plot_file(ortho_input(_g),   ax=axes[0], title=f"Before — uncorrected (group {_g!r})")
vz.plot_file(_gp0["corrected"], ax=axes[1], title=f"After — colour-corrected (group {_g!r})")
plt.tight_layout(); plt.show()

## Step 3: Interactive Map

In [ ]:
vz.plot_file(group_paths(discover_groups()[0])["corrected"])

## Steps performed

1. **Percentile stretch** — each RGB band clipped to its 2nd–98th percentile across
   non-black pixels and rescaled to 0–255 uint8.
2. **Before / after comparison** — matplotlib side-by-side render.
3. **Interactive map** — Folium overlay on OpenStreetMap.

**Next:** Part 3 converts the corrected GeoTIFF to Cloud-Optimised GeoTIFF (COG) layout
using the **GeoBrix `cog_gbx` writer**.

**Re-run behaviour:** each group is checkpointed under `_checkpoint.json`. A re-run skips any group whose corrected output already matches the input signature. Set `FORCE_CORRECTED = True` in `config_nb` to recompute regardless.